In [5]:
from collections import Counter, defaultdict
from matplotlib.lines import Line2D
from scipy.spatial import distance
import matplotlib.pyplot as plt
from rdkit import Chem
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
import json
import os
import tarfile
import pymol
from pymol import cmd
import matplotlib.image as mpimg
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen
from rdkit.Chem.QED import qed
import matplotlib.patches as patches
from pycirclize import Circos
from itertools import combinations
from matplotlib.colors import to_hex
import numpy as np
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import stylia

# Format: slide | Style: ersilia — change with stylia.set_format() / stylia.set_style()
stylia.set_format("print")
stylia.set_style("article")

In [6]:
DOCKING_RESULTS_ORIGINAL = {}

# Define some paths
root = '../../../github/mtb-targeted-protein-degradation/notebooks'
PATH_TO_DOCKING_RESULTS_ORIGINAL = os.path.join(root, "..", "output", "unidock_docking", 'docking_results')
PATH_TO_DOCKING_RESULTS_REAL_2 = os.path.join(root, "..", "output", "unidock_REAL_docking_2", 'docking_results')

# Mapping IDs to SMILES
ID_TO_SMILES = pickle.load(open(os.path.join(root, "..", "output", "enamine_characterization", "ID_TO_SMI.pkl"), "rb"))

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "output", "pocket_detection_data.csv"))
pocket_detection_data_interpro = pd.read_csv(os.path.join(root, "..", "output", "pocket_detection_data_interpro.tsv"), sep='\t')

# Uniprot to gene name
uniprot_to_gene = pd.read_csv(os.path.join(root, "..", "data", "mtb_trna_synthetases_bosch_2021_fig5.csv"))
uniprot_to_gene = {i: j for i,j in zip(uniprot_to_gene['uniprot_ac'], uniprot_to_gene['gene_name_in_bosch_2021'])}

# For each pocket - ORIGINAL
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_ORIGINAL))):
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_ORIGINAL, pocket, 'report.csv'))
    DOCKING_RESULTS_ORIGINAL[pocket] = {i: j for i, j in zip(scores['compound'], scores['score'])}

# Define pockets, compounds and proteins
POCKETS = sorted(DOCKING_RESULTS_ORIGINAL)
COMPOUNDS = sorted(DOCKING_RESULTS_ORIGINAL[POCKETS[0]])
PROTEINS = sorted(set([i.split("_")[1] for i in POCKETS]))
print(f"Number of pockets: {len(POCKETS)}")
print(f"Number of compounds: {len(COMPOUNDS)}")
print(f"Number of proteins: {len(PROTEINS)}")

100%|██████████| 276/276 [00:13<00:00, 20.35it/s]

Number of pockets: 276
Number of compounds: 100154
Number of proteins: 21


In [7]:
# Get minimum docking scores per protein
DOCKING_RESULTS_ORIGINAL_PROTEINS = {i: defaultdict(int) for i in PROTEINS}
for pocket in tqdm(POCKETS):
    protein = pocket.split("_")[1]
    for cpd in sorted(DOCKING_RESULTS_ORIGINAL[pocket]):
        DOCKING_RESULTS_ORIGINAL_PROTEINS[protein][cpd] = min(DOCKING_RESULTS_ORIGINAL_PROTEINS[protein][cpd], DOCKING_RESULTS_ORIGINAL[pocket][cpd])

  0%|          | 0/276 [00:00<?, ?it/s]

100%|██████████| 276/276 [00:12<00:00, 22.87it/s]


In [ ]:
# Color palette taken from the figure_1 color mapping (gene -> hex color), instead of tab20/tab20b
with open(os.path.join("..", "output", "plots", "figure_1", "color_mapping.json")) as f:
    color_mapping = json.load(f)
cmap_dict = dict(color_mapping["gene_to_color"])

def get_hit_overlap(h1, h2):
        return len(h1.intersection(h2))

fig, axs = stylia.create_figure(1, 4)

for count, SCORE_CUTOFF in enumerate([-8, -9, -10, -11]):
    ax = axs.next()

    proteins = sorted(set([p.split("_")[1] for p in DOCKING_RESULTS_ORIGINAL]))
    proteins_to_hits = {p: set() for p in proteins} 

    for pocket in sorted(DOCKING_RESULTS_ORIGINAL):
        p = pocket.split("_")[1]
        hits = [cpd for cpd, score in DOCKING_RESULTS_ORIGINAL[pocket].items() if score < SCORE_CUTOFF]
        proteins_to_hits[p].update(hits)

    HIT_OVERLAP = {}
    for c1, p1 in enumerate(proteins):
        for p2 in proteins[c1:]:
            ov = get_hit_overlap(proteins_to_hits[p1], proteins_to_hits[p2])
            HIT_OVERLAP[(p1, p2)] = ov
            HIT_OVERLAP[(p2, p1)] = ov

    row_names = proteins
    matrix = [[HIT_OVERLAP[(i, j)] for j in row_names] for i in row_names]
    matrix_df = pd.DataFrame(matrix, index=row_names, columns=row_names)
    np.fill_diagonal(matrix_df.values, 0)
    node_strength = matrix_df.sum(axis=0) + matrix_df.sum(axis=1)
    order = node_strength.sort_values(ascending=False).index
    matrix_df = matrix_df.loc[order, order]
    matrix_df_genes = matrix_df.rename(index=uniprot_to_gene).rename(columns=uniprot_to_gene)

    node_strength = (matrix_df_genes.sum(axis=0) + matrix_df_genes.sum(axis=1)) / 2
    total_sum = matrix_df_genes.values.sum() / 2
    thr = total_sum / 25
    rename_map = {}
    c = 1
    for g in matrix_df_genes.index:
        if node_strength.loc[g] < thr:
            rename_map[g] = " " * c
            cmap_dict[" " * c] = cmap_dict[g]
            c += 1
    matrix_df_plot = matrix_df_genes.rename(index=rename_map).rename(columns=rename_map)
    matrix_df_plot.values[np.tril_indices_from(matrix_df_plot, k=1)] = 0

    circos = Circos.chord_diagram(
        matrix_df_plot,
        space=1,
        cmap=cmap_dict,
        label_kws=dict(size=15),
        link_kws=dict(ec="k", lw=0.5, alpha=0.8))

    plot = circos.plotfig()

    canvas = FigureCanvas(plot)
    canvas.draw()
    w, h = plot.canvas.get_width_height()
    img = np.frombuffer(canvas.tostring_argb(), dtype=np.uint8).reshape(h, w, 4)[:, :, [1,2,3,0]] 

    ax.imshow(img)
    ax.axis("off")
    stylia.label(ax, xlabel="", ylabel="", title=f"Docking score cut-off: {SCORE_CUTOFF}")#\n[count: {int(total_sum)}]")
    plt.close(plot)

stylia.save_figure(os.path.join("..", "output", "plots", "figure_1", "hit_overlap_circos_stylia.png"))